In [6]:
import pandas as pd
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# =========================
# CONFIG
# =========================
model_name = "deepseek-ai/deepseek-llm-7b-chat"
input_file = "final_combined_dataset.csv"
output_file = "abstention_deepseek_xml.csv"
save_every = 50
batch_size = 6

In [7]:
# =========================
# LOAD MODEL
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding_side="left",
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token  # IMPORTANT

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/accelerate/utils/modeling.py:1582: UserWarning: Current model requires 3968 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(102400, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (n

In [8]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv(input_file)

# Resume logic
if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    start_idx = len(df_existing)
    print(f" Resuming from row {start_idx}")
else:
    start_idx = 0


# =========================
# PROMPT (XML FORMAT)
# =========================
def build_prompt(row):
    return f"""
You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>Give 2-3 line explanation</reasoning>

If format is violated, output:
<answer>idk</answer>
<reasoning>Format error</reasoning>
"""


In [9]:
# =========================
# PARSER (XML SAFE)
# =========================
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    try:
        ans_match = re.search(r"<answer>\s*(.*?)\s*</answer>", output, re.IGNORECASE)
        if ans_match:
            final_answer = ans_match.group(1).strip().lower()

        reason_match = re.search(
            r"<reasoning>\s*(.*?)\s*</reasoning>",
            output,
            re.IGNORECASE | re.DOTALL
        )
        if reason_match:
            reasoning = reason_match.group(1).strip()

    except:
        pass

    # Safety checks
    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning

In [ ]:
# =========================
# INFERENCE LOOP
# =========================
results_buffer = []

for i in tqdm(range(start_idx, len(df), batch_size), desc="Running Inference"):

    batch = df.iloc[i:i+batch_size]
    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.0,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for prompt, full_output in zip(prompts, decoded):
        cleaned = full_output[len(prompt):].strip()

        final_answer, reasoning = parse_output(cleaned)

        results_buffer.append({
            "pred_final_answer": final_answer,
            "pred_reasoning": reasoning
        })

    # =========================
    # SAVE EVERY 50 ROWS
    # =========================
    if (i + batch_size) % save_every == 0 or (i + batch_size) >= len(df):

        start_save_idx = i - len(results_buffer) + batch_size
        df_chunk = df.iloc[start_save_idx:i + batch_size].copy()

        df_chunk["pred_final_answer"] = [r["pred_final_answer"] for r in results_buffer]
        df_chunk["pred_reasoning"] = [r["pred_reasoning"] for r in results_buffer]

        if os.path.exists(output_file):
            df_chunk.to_csv(output_file, mode="a", index=False, header=False)
        else:
            df_chunk.to_csv(output_file, index=False)

        print(f" Saved up to row {i + batch_size}")

        results_buffer = []  # reset buffer

print(" Done! DeepSeek pipeline completed.")

Running Inference:   0%|          | 0/587 [00:00<?, ?it/s]/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
